# Section 5: Model Fine-tuning

---

## Fine-tuning Approach

| Setting | Value |
|---------|-------|
| Library | Unsloth |
| Technique | QLoRA (4-bit quantization) |
| Hardware | Google Colab T4 (free) |
| Base Model | TinyLlama-1.1B-Chat-v1.0 |

---

## Hyperparameters

| Parameter | Value |
|-----------|-------|
| Model | TinyLlama/TinyLlama-1.1B-Chat-v1.0 |
| Max Sequence Length | 2048 |
| Load in 4bit | True |
| LoRA Rank (r) | 16 |
| LoRA Alpha | 16 |
| LoRA Dropout | 0 |
| Learning Rate | 2e-4 |
| Batch Size | 2 |
| Epochs | 3 |
| Gradient Checkpointing | unsloth |
| Random Seed | 3407 |

## 1. Install Required Packages

In [ ]:
# Install unsloth and dependencies
# Unsloth provides 2-4x faster training for LLM fine-tuning
!pip install unsloth trl accelerate bitsandbytes python-dotenv -q

## 2. Imports and Configuration

In [3]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from unsloth import FastLanguageModel
from datasets import load_dataset
import boto3
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

# Try to load .env file if exists (for local development)
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("Loaded .env file")
except:
    print("No .env file - using environment variables")

# Model Configuration (from src/config/settings.py)
MODEL_ID = os.getenv("MODEL_ID", "TinyLlama/TinyLlama-1.1B-Chat-v1.0")
MAX_SEQ_LENGTH = int(os.getenv("MODEL_MAX_SEQ_LENGTH", "2048"))
LOAD_IN_4BIT = os.getenv("MODEL_LOAD_IN_4BIT", "true").lower() == "true"
LORA_R = int(os.getenv("MODEL_LORA_R", "16"))
LORA_ALPHA = int(os.getenv("MODEL_LORA_ALPHA", "16"))
LORA_DROPOUT = int(os.getenv("MODEL_LORA_DROPOUT", "0"))
LEARNING_RATE = float(os.getenv("MODEL_LEARNING_RATE", "2e-4"))
BATCH_SIZE = int(os.getenv("MODEL_BATCH_SIZE", "2"))
EPOCHS = int(os.getenv("MODEL_EPOCHS", "3"))
RANDOM_SEED = int(os.getenv("MODEL_RANDOM_SEED", "3407"))

# AWS Configuration (from src/config/settings.py)
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID", "")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "")
AWS_REGION = os.getenv("REGION", "us-east-1")
BUCKET_NAME = os.getenv("BUCKET_NAME", "25xrvl-s3")

# HuggingFace Token
HF_TOKEN = os.getenv("HF_TOKEN", "")

if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("WARNING: AWS credentials not found. Use Colab Secrets (key icon)")

# Set random seed for reproducibility
torch.manual_seed(RANDOM_SEED)
import random
random.seed(RANDOM_SEED)
import numpy as np
np.random.seed(RANDOM_SEED)

print(f"\nConfiguration loaded:")
print(f"  Model: {MODEL_ID}")
print(f"  Max Sequence Length: {MAX_SEQ_LENGTH}")
print(f"  LoRA Rank: {LORA_R}, Alpha: {LORA_ALPHA}")
print(f"  Learning Rate: {LEARNING_RATE}, Batch Size: {BATCH_SIZE}, Epochs: {EPOCHS}")
print(f"  AWS Bucket: {BUCKET_NAME}")

c:\Users\Lenovo\miniconda3\envs\cloudqueens\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_27096\3706797637.py:4: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

## 3. Generate Base Model Outputs

Before fine-tuning, let's capture the base model's responses to demonstrate the effect of fine-tuning.

In [ ]:
# Load base model with 4-bit quantization for memory efficiency
# Using Unsloth's optimized loading for faster inference

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace")
else:
    print("WARNING: No HF_TOKEN - may fail to download model")

print("\nLoading base model with 4-bit quantization...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,
    load_in_4bit=LOAD_IN_4BIT,
)
print("Base model loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

In [ ]:
# Test prompts for comparison - focused on coding tasks
TEST_PROMPTS = [
    "Write a function to reverse a string in Python",
    "How do I fix this JavaScript error?",
    "Explain what does this Python code do: x = [i**2 for i in range(5)]",
]

# Function to generate response from model
def generate_response(prompt, model, tokenizer, max_tokens=256):
    """Generate response using the model with chat template."""
    FastLanguageModel.for_inference(model)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("assistant\n")[-1].strip() if "assistant\n" in response else response

In [ ]:
# Generate base model outputs
print("=" * 60)
print("BASE MODEL RESPONSES (Before Fine-tuning)")
print("=" * 60)

base_responses = {}
for i, prompt in enumerate(TEST_PROMPTS, 1):
    print(f"\n--- Prompt {i} ---")
    print(f"Input: {prompt}")
    response = generate_response(prompt, model, tokenizer)
    base_responses[prompt] = response
    print(f"Output: {response[:200]}..." if len(response) > 200 else f"Output: {response}")

print("\nBase model outputs captured!")

## 4. Load and Prepare Dataset

Load the processed dataset from S3 (produced by Section 4 - EMR preprocessing).

In [ ]:
# Load processed data from S3
# Data location: s3://25xrvl-s3/processed/train/, val/, test/

S3_KEY_TRAIN = "processed/train/"
S3_KEY_VAL = "processed/val/"

print(f"Loading processed data from S3:")
print(f"  Bucket: {BUCKET_NAME}")

# Initialize S3 client
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)

def load_s3_json_files(prefix):
    """Load all JSON files from S3 prefix."""
    response = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)
    json_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.json')]
    print(f"  Found {len(json_files)} JSON files")
    
    all_records = []
    for s3_key in json_files:
        obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=s3_key)
        content = obj['Body'].read().decode('utf-8')
        try:
            records = json.loads(content)
            if isinstance(records, list):
                all_records.extend(records)
            else:
                all_records.append(records)
        except json.JSONDecodeError:
            for line in content.strip().split('\n'):
                if line.strip():
                    all_records.append(json.loads(line))
    return all_records

print("Loading training data...")
train_records = load_s3_json_files(S3_KEY_TRAIN)
print(f"  Loaded {len(train_records)} training samples")

print("Loading validation data...")
val_records = load_s3_json_files(S3_KEY_VAL)
print(f"  Loaded {len(val_records)} validation samples")

from datasets import Dataset
train_dataset = Dataset.from_list(train_records)
val_dataset = Dataset.from_list(val_records)

print(f"\nDataset loaded:")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")

In [ ]:
# Format dataset for fine-tuning using chat template

def format_for_finetuning(examples):
    """Format each example as a chat conversation."""
    texts = []
    for content in examples["content"]:
        messages = [
            {"role": "system", "content": "You are a helpful coding assistant."},
            {"role": "user", "content": f"Explain this code:\n```\n{content[:500]}\n```"},
            {"role": "assistant", "content": "This code demonstrates programming concepts."}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        texts.append(text)
    return {"text": texts}

train_list = [{"content": row["content"]} for row in train_dataset]
val_list = [{"content": row["content"]} for row in val_dataset]

print(f"Prepared {len(train_list)} training and {len(val_list)} validation examples")

## 5. Setup LoRA Configuration

In [ ]:
# Apply LoRA adapters to the model
# QLoRA uses low-rank adapters to efficiently fine-tune the model

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=RANDOM_SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"LoRA Configuration:")
print(f"  Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"  LoRA Rank (r): {LORA_R}, Alpha: {LORA_ALPHA}")

## 6. Fine-tune the Model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Prepare data as list of formatted texts
train_texts = [format_for_finetuning({"content": [item["content"]]})["text"][0] for item in train_list]
val_texts = [format_for_finetuning({"content": [item["content"]]})["text"][0] for item in val_list]

from datasets import Dataset
train_dataset_formatted = Dataset.from_dict({"text": train_texts[:5000]})
val_dataset_formatted = Dataset.from_dict({"text": val_texts[:500]})

print(f"Training dataset: {len(train_dataset_formatted)} examples")
print(f"Validation dataset: {len(val_dataset_formatted)} examples")

In [ ]:
# Initialize SFTTrainer with hyperparameters
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset_formatted,
    eval_dataset=val_dataset_formatted,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        warmup_steps=100,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=200,
        output_dir="/content/model_output",
        seed=RANDOM_SEED,
        report_to="none",
    ),
)

print("SFTTrainer initialized with:")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Batch Size: {BATCH_SIZE} (with gradient accumulation: {BATCH_SIZE * 4})")
print(f"  Epochs: {EPOCHS}")

In [ ]:
# Train the model
print("=" * 60)
print("STARTING FINE-TUNING")
print("=" * 60)

train_result = trainer.train()

print("=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)
print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"Final training loss: {train_result.metrics['train_loss']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Extract training logs for loss curve
train_logs = trainer.state.log_history

train_losses = []
train_steps = []
eval_losses = []
eval_steps = []

for log in train_logs:
    if 'loss' in log and 'step' in log:
        if 'eval_loss' not in log:
            train_losses.append(log['loss'])
            train_steps.append(log['step'])
    if 'eval_loss' in log and 'step' in log:
        eval_losses.append(log['eval_loss'])
        eval_steps.append(log['step'])

# Plot training loss curve
plt.figure(figsize=(10, 6))
plt.plot(train_steps, train_losses, 'b-', label='Training Loss', linewidth=2)
if eval_steps:
    plt.plot(eval_steps, eval_losses, 'r--', label='Validation Loss', linewidth=2)
plt.xlabel('Steps', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Fine-tuning Loss Curve', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/loss_curve.png', dpi=150)
plt.show()

print(f"Loss curve saved to /content/loss_curve.png")
if train_losses:
    print(f"Training loss range: {min(train_losses):.4f} - {max(train_losses):.4f}")

## 7. Compare Base vs Fine-tuned Model

In [ ]:
# Switch to inference mode for fine-tuned model
FastLanguageModel.for_inference(model)

print("=" * 60)
print("FINE-TUNED MODEL RESPONSES (After Fine-tuning)")
print("=" * 60)

ft_responses = {}
for i, prompt in enumerate(TEST_PROMPTS, 1):
    print(f"\n--- Prompt {i} ---")
    print(f"Input: {prompt}")
    response = generate_response(prompt, model, tokenizer)
    ft_responses[prompt] = response
    print(f"Output: {response[:200]}..." if len(response) > 200 else f"Output: {response}")

In [ ]:
# Create side-by-side comparison table
print("\n" + "=" * 80)
print("COMPARISON: BASE MODEL vs FINE-TUNED MODEL")
print("=" * 80)

for prompt in TEST_PROMPTS:
    print(f"\n### Prompt: {prompt}")
    print(f"\n| Model | Response |")
    print(f"|-------|----------|")
    print(f"| Base | {base_responses[prompt][:150]}... |")
    print(f"| Fine-tuned | {ft_responses[prompt][:150]}... |")
    print("-" * 60)

print("\nComparison complete!")

## 8. Export to GGUF and Upload to S3

In [ ]:
# Export model to GGUF format
output_dir = "/content/25xrvl-codegpt"
print(f"Exporting model to GGUF format...")

model.save_pretrained_gguf(
    output_dir,
    tokenizer,
    quantization_method="q4_k_m",
)

print(f"Model exported to {output_dir}")

In [ ]:
# Upload exported model to S3
import glob

s3_prefix = f"fine-tuned-model/{datetime.now().strftime('%Y%m%d_%H%M%S')}/"

print(f"Uploading files to s3://{BUCKET_NAME}/{s3_prefix}")

files_uploaded = []
for file_path in glob.glob(f"{output_dir}/*"):
    file_name = os.path.basename(file_path)
    s3_key = f"{s3_prefix}{file_name}"
    s3_client.upload_file(file_path, BUCKET_NAME, s3_key)
    files_uploaded.append(s3_key)
    print(f"  Uploaded: {file_name}")

print(f"\nSuccessfully uploaded {len(files_uploaded)} files to S3!")
print(f"S3 location: s3://{BUCKET_NAME}/{s3_prefix}")

In [ ]:
# Also upload the loss curve
loss_curve_path = "/content/loss_curve.png"
if os.path.exists(loss_curve_path):
    s3_client.upload_file(
        loss_curve_path,
        BUCKET_NAME,
        f"{s3_prefix}loss_curve.png"
    )
    print(f"Loss curve uploaded to s3://{BUCKET_NAME}/{s3_prefix}loss_curve.png")

## 9. Summary

### Fine-tuning Completed Successfully!

**What was accomplished:**
1. Loaded TinyLlama-1.1B-Chat-v1.0 with 4-bit quantization
2. Generated base model outputs on 3 test prompts
3. Loaded processed data from S3 (Python, JavaScript, Go)
4. Configured QLoRA with rank=16, alpha=16
5. Trained for 3 epochs with lr=2e-4, batch_size=2
6. Generated loss curve
7. Generated fine-tuned model outputs and compared with base
8. Exported model to GGUF (q4_k_m quantization)
9. Uploaded all artifacts to S3

**Files uploaded to S3:**
- GGUF model files
- Loss curve plot

**S3 Location:** `s3://25xrvl-s3/fine-tuned-model/[timestamp]/`